# 🧠 Module 4: Face-to-Body Association Pipeline
**Architecture:** ArcFace + CLIP-ReID + True Self-Adaptive Gallery + Dual Pinecone Indices
**Pipeline:** Face/Body Spatial Merge → Face Ground Truth Query → Retroactive Identity Healing → True Drop Cost Gallery Optimization

### 🛠️ Cell 1: Environment Setup

In [ ]:
%%capture
!pip install ultralytics boxmot opencv-python scipy
!pip install pinecone imagekitio scikit-learn pymongo
!pip install insightface onnxruntime-gpu

print("Dependencies Ready. Added InsightFace for Enterprise Face Recognition.")

### ⚙️ Cell 2: Global Configuration & Parameters

In [ ]:
import os
import shutil
import cv2
import time
import torch
import numpy as np
import pickle
import concurrent.futures
from pathlib import Path
from tqdm.auto import tqdm
import torch.nn.functional as F
from sklearn.cluster import DBSCAN
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient

# ══════════════════════════════════════════════════════════════
# 1. PATHS & CREDENTIALS
# ══════════════════════════════════════════════════════════════
INPUT_VIDEO   = "/kaggle/input/datasets/abderrahimzineddine/reid-testing-samples/store_cam1_mirrored.mp4" # Face Choke Point Camera
OUTPUT_DIR    = Path("/kaggle/working/module2_database")
DB_IMAGES_DIR = OUTPUT_DIR / "saved_identities"

try:
    secrets = UserSecretsClient()
    IMAGEKIT_URL_ENDPOINT = secrets.get_secret("IMAGEKIT_URL_ENDPOINT")
    IMAGEKIT_PUBLIC_KEY   = secrets.get_secret("IMAGEKIT_PUBLIC_KEY")
    IMAGEKIT_PRIVATE_KEY  = secrets.get_secret("IMAGEKIT_PRIVATE_KEY")
    MONGO_URI             = secrets.get_secret("MONGO_URI")
    PINECONE_API_KEY      = secrets.get_secret("PINECONE_API_KEY")
except:
    IMAGEKIT_PRIVATE_KEY  = "YOUR_IMAGEKIT_PRIVATE"
    MONGO_URI             = "YOUR_MONGO_URI"
    PINECONE_API_KEY      = "YOUR_PINECONE_KEY"

PINECONE_INDEX_NAME = "supermarket-reid"
PINECONE_FACE_INDEX_NAME = "supermarket-faces" # <--- Added the new dedicated Face Index
RESET_DATABASE = False   

# ── SMART PKL / DATASET DETECTION ──
input_pkls = list(Path('/kaggle/input').rglob('identity_buffer*.pkl'))
if input_pkls and not RESET_DATABASE:
    LOAD_BUFFER_PATH = input_pkls[0]
    print(f"Found imported memory buffer: {LOAD_BUFFER_PATH}")
else:
    LOAD_BUFFER_PATH = Path('/kaggle/working/identity_buffer.pkl')

SAVE_BUFFER_PATH = Path('/kaggle/working/identity_buffer.pkl')

if RESET_DATABASE:
    if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
    if SAVE_BUFFER_PATH.exists(): SAVE_BUFFER_PATH.unlink()

DB_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════
# 2. PAPER 1 ALGORITHM THRESHOLDS
# ══════════════════════════════════════════════════════════════
REID_CONF_THRESHOLD = 0.75   
POSE_CONF_THRESHOLD = 0.3    
MIN_VISIBLE_RATIO   = 0.5    

# Face specific
FACE_CONF_THRESHOLD = 0.5
FACE_MATCH_THRESHOLD = 0.55

TAU          = 2.0   
MIN_COS_SIM  = 0.75  
MAX_IMAGES   = 50    
UDM_MIN_SIZE = 5     
UDM_EPSILON  = 0.15  

print("Configuration Loaded. Folders Ready.")

### 🧠 Cell 3: Model Initialization & Cloud Connection

In [ ]:
from ultralytics import RTDETR, YOLO
from boxmot.trackers.botsort.botsort import BotSort
from imagekitio import ImageKit
from pinecone import Pinecone
import boxmot
import pkgutil
import inspect
from insightface.app import FaceAnalysis

n_gpus = torch.cuda.device_count()
DET_DEVICE  = 'cuda:0'
POSE_DEVICE = 'cuda:1' if n_gpus > 1 else 'cuda:0'
REID_DEVICE = 'cuda:1' if n_gpus > 1 else 'cuda:0'

print("Loading Models...")
detector = RTDETR('rtdetr-l.pt').to(DET_DEVICE)
tracker = BotSort(reid_weights='osnet_x0_25_msmt17.pt', device=DET_DEVICE, half=False)
pose_model = YOLO('yolo26s-pose.pt').to(POSE_DEVICE)

ReID = None
for importer, modname, ispkg in pkgutil.walk_packages(boxmot.__path__, boxmot.__name__ + '.'):
    try:
        mod = __import__(modname, fromlist="dummy")
        for name, obj in inspect.getmembers(mod):
            if inspect.isclass(obj) and name in ['ReID', 'ReIDDetectMultiBackend']:
                ReID = obj; break
    except Exception: continue
    if ReID is not None: break

reid_model = ReID(weights='clip_market1501.pt', device=REID_DEVICE, half=False)

print("Loading Face Engine (InsightFace/ArcFace)...")
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

print("Connecting to Databases...")
imagekit = ImageKit(private_key=IMAGEKIT_PRIVATE_KEY)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
face_index = pc.Index(PINECONE_FACE_INDEX_NAME) # <--- Initialized the Face Index

mongo_client = MongoClient(MONGO_URI)
db = mongo_client["supermarket_db"]
persons_col = db["persons"]

if RESET_DATABASE:
    persons_col.delete_many({}) 
    print("MongoDB cleared.")

### 🏗️ Cell 4: The Database Architecture Classes (Upgraded)

In [ ]:
class SightingTracker:
    def __init__(self, fps=30):
        self.fps = fps
        self.active_sightings = {} 
        self.completed_sightings = []

    def _frame_to_ts(self, frame):
        return int(time.time()) + (frame / self.fps)

    def update_track(self, tid, frame, conf, ratio, img, gid=None):
        if tid not in self.active_sightings:
            self.active_sightings[tid] = {
                "camera_id": "Face_Cam_Entrance",
                "track_id": tid,
                "global_id": gid,
                "entry_ts": self._frame_to_ts(frame),
                "exit_ts": self._frame_to_ts(frame),
                "duration_sec": 0.0,
                "frame_start": frame,
                "frame_end": frame,
                "n_detections": 1,
                "sum_det_score": float(conf),
                "sum_skeleton_ratio": float(ratio),
                "best_crop_quality": float(conf * ratio),
                "best_crop_img": img,
                "best_face_img": None,
                "best_face_emb": None,
                "best_face_conf": 0.0
            }
        else:
            s = self.active_sightings[tid]
            if gid is not None and s["global_id"] is None:
                s["global_id"] = gid
            s["exit_ts"] = self._frame_to_ts(frame)
            s["frame_end"] = frame
            s["duration_sec"] = s["exit_ts"] - s["entry_ts"]
            s["n_detections"] += 1
            s["sum_det_score"] += float(conf)
            s["sum_skeleton_ratio"] += float(ratio)
            
            q = float(conf * ratio)
            if q > s["best_crop_quality"]:
                s["best_crop_quality"] = q
                s["best_crop_img"] = img

    def update_face(self, tid, face_img, face_emb, face_conf):
        s = self.active_sightings.get(tid)
        if s and float(face_conf) > s["best_face_conf"]:
            s["best_face_conf"] = float(face_conf) # Strip PyMongo-breaking np.float32 types
            s["best_face_img"] = face_img
            s["best_face_emb"] = face_emb

    def assign_global_id(self, tid, gid):
        if tid in self.active_sightings:
            self.active_sightings[tid]["global_id"] = gid

    def close_track(self, tid):
        if tid in self.active_sightings:
            s = self.active_sightings.pop(tid)
            if s["global_id"] is not None:
                s["avg_det_score"] = float(s["sum_det_score"] / s["n_detections"])
                s["avg_skeleton_ratio"] = float(s["sum_skeleton_ratio"] / s["n_detections"])
                del s["sum_det_score"]
                del s["sum_skeleton_ratio"]
                self.completed_sightings.append(s)


class UnknownDataManager:
    def __init__(self, eps, min_samples):
        self.pool = [] 
        self.eps = eps
        self.min_samples = min_samples

    def add(self, item):
        self.pool.append(item)

    def run_dbscan(self, gallery):
        if len(self.pool) < self.min_samples:
            return []
        
        embs = torch.stack([x['emb'] for x in self.pool])
        sim_matrix = torch.mm(embs, embs.t()).cpu().numpy()
        dist_matrix = np.clip(1.0 - sim_matrix, 0.0, 2.0)
        
        clustering = DBSCAN(eps=self.eps, min_samples=self.min_samples, metric='precomputed').fit(dist_matrix)
        labels = clustering.labels_
        
        actions = []
        new_pool = []
        
        for lbl in set(labels):
            if lbl == -1: continue
            indices = np.where(labels == lbl)[0]
            cluster_items = [self.pool[i] for i in indices]
            cluster_track_ids = list(set([x['track_id'] for x in cluster_items]))
            
            # --- THE UDM BYPASS ---
            # If the Face Camera recognized ANY frame in this cluster, we force the Face ID!
            forced_gids = [x.get('forced_gid') for x in cluster_items if x.get('forced_gid') is not None]
            if forced_gids:
                best_gid = max(set(forced_gids), key=forced_gids.count)
                for item in cluster_items:
                    gallery.add_observation(best_gid, item)
                actions.append(("FACE_VIP_BYPASS", best_gid, len(cluster_items), cluster_track_ids))
                continue

            cluster_mean = F.normalize(torch.mean(torch.stack([x['emb'] for x in cluster_items]), dim=0), p=2, dim=0)
            
            merged = False
            if gallery.num_classes() > 0:
                centroids = gallery.get_weighted_centroids()
                sims = torch.mv(centroids, cluster_mean)
                best_sim, best_idx = torch.max(sims, dim=0)
                
                if (1.0 - best_sim.item()) <= self.eps:
                    best_gid = gallery.gids[best_idx.item()]
                    for item in cluster_items:
                        gallery.add_observation(best_gid, item)
                    actions.append(("MERGE", best_gid, len(cluster_items), cluster_track_ids))
                    merged = True
            
            if not merged:
                new_gid = gallery.mint_new_class()
                for item in cluster_items:
                    gallery.add_observation(new_gid, item)
                actions.append(("MINT", new_gid, len(cluster_items), cluster_track_ids))
                
        for i, lbl in enumerate(labels):
            if lbl == -1: new_pool.append(self.pool[i])
        self.pool = new_pool
        return actions


class GalleryManager:
    def __init__(self, max_images):
        self.classes = {} 
        self.gids = []
        self.max_images = max_images
        self.current_max_id = 0

    def num_classes(self):
        return len(self.gids)

    def mint_new_class(self):
        self.current_max_id += 1
        gid = self.current_max_id
        self.classes[gid] = {'embs':[], 'ratios':[], 'imgs':[], 'filepaths':[], 'scores':[]}
        self.gids.append(gid)
        (DB_IMAGES_DIR / f"global_id_{gid:03d}").mkdir(exist_ok=True)
        return gid

    def get_weighted_centroids(self):
        centroids = []
        for gid in self.gids:
            data = self.classes[gid]
            embs = torch.stack(data['embs'])
            ratios = torch.tensor(data['ratios'], device=embs.device).unsqueeze(1)
            w_sum = torch.sum(embs * ratios, dim=0)
            centroids.append(F.normalize(w_sum, p=2, dim=0))
        return torch.stack(centroids)

    def classify_live_track(self, emb, tau, exclude_gids=None):
        if self.num_classes() == 0: return None, 0.0, 0.0
            
        centroids = self.get_weighted_centroids()
        sims = torch.mv(centroids, emb)
        
        if exclude_gids:
            for i, gid in enumerate(self.gids):
                if gid in exclude_gids: sims[i] = -float('inf') 
        
        probs = F.softmax(sims * 15.0, dim=0)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        
        max_prob = sorted_probs[0].item()
        best_sim = sims[sorted_idx[0]].item()
        best_gid = self.gids[sorted_idx[0].item()]
        
        if best_sim == -float('inf'): return None, 0.0, 0.0
        if best_sim < MIN_COS_SIM: return best_gid, 0.0, max_prob
            
        if len(sorted_probs) > 1: ratio = max_prob / (sorted_probs[1].item() + 1e-9)
        else: ratio = float('inf')
            
        return best_gid, ratio, max_prob

    def add_observation(self, gid, item):
        data = self.classes[gid]
        data['embs'].append(item['emb'])
        data['ratios'].append(item['ratio'])
        data['scores'].append(item['score'])
        
        filename = f"frame_{item['frame']:06d}_trk_{item['track_id']:03d}.jpg"
        filepath = DB_IMAGES_DIR / f"global_id_{gid:03d}" / filename
        cv2.imwrite(str(filepath), item['img'])
        data['filepaths'].append(filepath)
        self._optimize_budget(gid)

    # ── THE FIX: True Self-Adaptive Gallery Drop Cost Formula ──
    def _optimize_budget(self, gid):
        data = self.classes[gid]
        if len(data['embs']) <= self.max_images: return
        
        embs = torch.stack(data['embs'])
        
        # 1. Redundancy: How similar is this crop to the OTHER 50 crops of John?
        sim_matrix = torch.mm(embs, embs.t())
        sim_matrix.fill_diagonal_(-1.0) 
        redundancy, _ = torch.max(sim_matrix, dim=1)
        
        # 2. Uncertainty (Entropy): How much does this crop look like OTHER people?
        centroids = self.get_weighted_centroids()
        if centroids.shape[0] > 1:
            sims_to_all = torch.mm(embs, centroids.t())
            probs = F.softmax(sims_to_all * 15.0, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=1)
            # Normalize entropy
            uncertainty = entropy / torch.log(torch.tensor(float(centroids.shape[1]), device=embs.device))
        else:
            uncertainty = torch.zeros(embs.shape[0], device=embs.device)
        
        # Drop Cost Formula: Drop the crop that is BOTH redundant and uncertain!
        cost = 0.5 * uncertainty.cpu().numpy() + 0.5 * redundancy.cpu().numpy()
        worst_idx = np.argmax(cost)
        
        worst_filepath = Path(data['filepaths'][worst_idx])
        # Only unlink if it's in the writable working directory (not imported read-only dataset)
        if worst_filepath.exists() and 'working' in str(worst_filepath): 
            worst_filepath.unlink() 
            
        for key in ['embs', 'ratios', 'scores', 'filepaths']:
            data[key].pop(worst_idx)

udm = UnknownDataManager(eps=UDM_EPSILON, min_samples=UDM_MIN_SIZE)
gallery = GalleryManager(max_images=MAX_IMAGES)
sighting_tracker = SightingTracker(fps=30)

def create_person_document(gid, first_seen_ts, col):
    if not col.find_one({"global_id": gid}):
        col.insert_one({
            "global_id": gid,
            "first_seen_ts": float(first_seen_ts),
            "last_seen_ts": float(first_seen_ts),
            "faces": [],
            "sightings": []
        })

### 🚀 Cell 5: Dual Face & Body Processing Loop

In [ ]:
def extract_reid_features(crops):
    with torch.no_grad():
        embeddings = reid_model(crops)
        if isinstance(embeddings, np.ndarray): embeddings = torch.tensor(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1).to(REID_DEVICE)
    return embeddings

def get_spatially_matched_face(body_box, faces):
    """Checks if a detected Face falls within the top 40% of a Body box."""
    x1, y1, x2, y2 = body_box
    body_top_half_y2 = y1 + (y2 - y1) * 0.4
    best_face, best_score = None, 0
    
    for f in faces:
        fx1, fy1, fx2, fy2 = f.bbox
        fcx, fcy = (fx1 + fx2) / 2, (fy1 + fy2) / 2
        if x1 <= fcx <= x2 and y1 <= fcy <= body_top_half_y2:
            if f.det_score > best_score:
                best_score = f.det_score
                best_face = f
    return best_face

def search_face_pinecone(face_emb_numpy):
    """Searches strictly the 'supermarket-faces' index for Ground Truth identity."""
    res = face_index.query(vector=face_emb_numpy.tolist(), top_k=1, include_metadata=True)
    if res.matches and res.matches[0].score > FACE_MATCH_THRESHOLD:
        return int(res.matches[0].metadata['global_id'])
    return None

def execute_identity_merge(target_gid, old_gid):
    """Retroactive Data Healing: Merges tracks, gallery arrays, and MongoDB records!"""
    tqdm.write(f"🧬 RETROACTIVE HEAL: Merging incorrect ID {old_gid} into True Face ID {target_gid}")
    
    for tid, gid in list(local_to_global_map.items()):
        if gid == old_gid:
            local_to_global_map[tid] = target_gid
            sighting_tracker.assign_global_id(tid, target_gid)
            
    if old_gid in gallery.classes:
        o_data = gallery.classes.pop(old_gid)
        gallery.gids.remove(old_gid)
        
        t_data = gallery.classes[target_gid]
        t_data['embs'].extend(o_data['embs'])
        t_data['ratios'].extend(o_data['ratios'])
        t_data['scores'].extend(o_data['scores'])
        
        target_dir = DB_IMAGES_DIR / f"global_id_{target_gid:03d}"
        for fp in o_data['filepaths']:
            fp = Path(fp)
            if fp.exists():
                new_fp = target_dir / fp.name
                # Avoid shutil.move crashing on read-only Kaggle inputs
                if 'input' in str(fp):
                    shutil.copy(str(fp), str(new_fp))
                else:
                    shutil.move(str(fp), str(new_fp))
                t_data['filepaths'].append(new_fp)
                
        # Instantly run true drop cost to prune back to 50
        while len(t_data['embs']) > gallery.max_images:
            gallery._optimize_budget(target_gid)
            
    old_doc = persons_col.find_one({"global_id": old_gid})
    if old_doc:
        persons_col.update_one(
            {"global_id": target_gid},
            {"$push": {
                "sightings": {"$each": old_doc.get("sightings", [])}, 
                "faces": {"$each": old_doc.get("faces", [])}
            }}
        )
        persons_col.delete_one({"global_id": old_gid})


if LOAD_BUFFER_PATH.exists() and not RESET_DATABASE:
    print(f"\n💧 Hydrating RAM from previous session ({LOAD_BUFFER_PATH})...")
    with open(LOAD_BUFFER_PATH, 'rb') as f: saved_data = pickle.load(f)
    
    # Check if images are mounted as a Kaggle dataset to resolve paths
    input_img_dirs = list(Path('/kaggle/input').rglob('saved_identities*'))
    input_img_base = input_img_dirs[0] if input_img_dirs else None
    
    max_id = 0
    for gid, data in saved_data.items():
        gallery.gids.append(gid)
        max_id = max(max_id, gid)
        
        valid_paths = []
        for p in data["filepaths"]:
            old_p = Path(p)
            if old_p.exists():
                valid_paths.append(old_p)
            elif input_img_base:
                # Map old absolute paths to the read-only dataset path
                new_p = input_img_base / old_p.parent.name / old_p.name
                if new_p.exists():
                    valid_paths.append(new_p)

        gallery.classes[gid] = {
            'embs': [torch.tensor(e).to(REID_DEVICE) for e in data["embeddings"]],
            'ratios': data["ratios"],
            'scores': data["det_scores"],
            'filepaths': valid_paths
        }
    gallery.current_max_id = max_id
    print(f"✅ Hydrated {gallery.num_classes()} body identities.")

cap = cv2.VideoCapture(INPUT_VIDEO)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
local_to_global_map = {}

with tqdm(total=TOTAL_FRAMES, desc="Multi-Modal Tracker", unit="fr") as pbar:
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        
        # 1. Body & Face Detection
        results = detector(frame, classes=[0], verbose=False)
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            pbar.update(1)
            continue
            
        faces_in_frame = face_app.get(frame)
        dets = torch.cat([boxes.xyxy, boxes.conf.unsqueeze(1), boxes.cls.unsqueeze(1)], dim=1).cpu().numpy()
        tracks = tracker.update(dets, frame)
        if tracks is None or len(tracks) == 0:
            pbar.update(1)
            continue
            
        valid_data = []
        for t in tracks:
            track_id = int(t[4])
            conf = float(t[5]) if len(t) > 5 else 1.0 
            if conf >= REID_CONF_THRESHOLD:
                x1, y1, x2, y2 = map(int, t[:4])
                x1, y1 = max(0, x1), max(0, y1)
                crop = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    valid_data.append({'track_id': track_id, 'conf': conf, 'box': [x1, y1, x2, y2], 'crop': crop})
            
        if valid_data:
            # 2. Body Pose Extraction
            pose_results = pose_model([d['crop'] for d in valid_data], verbose=False)
            pristine_data = []
            
            for i, res in enumerate(pose_results):
                visible_count = sum(1 for x, y, c in res.keypoints.data[0].cpu().numpy() if c > POSE_CONF_THRESHOLD) if res.keypoints is not None and len(res.keypoints.data) > 0 else 0
                ratio = visible_count / 17.0
                if ratio >= MIN_VISIBLE_RATIO:
                    pristine_data.append({**valid_data[i], 'ratio': ratio, 'score': valid_data[i]['conf']})
            
            if pristine_data:
                # 3. Body Embedding
                embeddings = extract_reid_features([d['crop'] for d in pristine_data])
                claimed_gids_this_frame = set()
                unresolved_items = []
                
                # 4. Identity Resolution (Face vs Body)
                for i, d in enumerate(pristine_data):
                    item = {'emb': embeddings[i], 'ratio': d['ratio'], 'score': d['score'], 'img': d['crop'], 'track_id': d['track_id'], 'frame': frame_count}
                    tid = d['track_id']
                    
                    # Ensure tracking container exists
                    gid = local_to_global_map.get(tid)
                    sighting_tracker.update_track(tid, frame_count, item['score'], item['ratio'], item['img'], gid)
                    
                    # Spatial Face Merge
                    matched_face = get_spatially_matched_face(d['box'], faces_in_frame)
                    face_truth_gid = None
                    
                    if matched_face and matched_face.det_score > FACE_CONF_THRESHOLD:
                        # Extract exact Face Crop
                        fx1, fy1, fx2, fy2 = map(int, matched_face.bbox)
                        face_img = frame[max(0, fy1):fy2, max(0, fx1):fx2]
                        
                        sighting_tracker.update_face(tid, face_img, matched_face.normed_embedding, matched_face.det_score)
                        face_truth_gid = search_face_pinecone(matched_face.normed_embedding)
                        
                        if face_truth_gid:
                            if gid and gid != face_truth_gid:
                                # Retroactive Data Healing Triggered
                                execute_identity_merge(face_truth_gid, gid)
                                gid = face_truth_gid
                            elif not gid:
                                # VIP UDM Bypass Marker
                                item['forced_gid'] = face_truth_gid
                                
                    # Route to correct Gallery Processor
                    gid = local_to_global_map.get(tid)
                    if gid is not None:
                        if gid not in claimed_gids_this_frame:
                            claimed_gids_this_frame.add(gid)
                            gallery.add_observation(gid, item)
                        else:
                            # Conflicting track, release to unknown
                            del local_to_global_map[tid]
                            unresolved_items.append(item)
                    else:
                        unresolved_items.append(item)
                
                # 5. Handle Unknowns
                for item in unresolved_items:
                    tid = item['track_id']
                    if gallery.num_classes() == 0:
                        new_gid = gallery.mint_new_class()
                        local_to_global_map[tid] = new_gid
                        claimed_gids_this_frame.add(new_gid)
                        gallery.add_observation(new_gid, item)
                        sighting_tracker.assign_global_id(tid, new_gid)
                        create_person_document(new_gid, sighting_tracker._frame_to_ts(frame_count), persons_col)
                    else:
                        if item.get('forced_gid'):
                            # Sent straight to UDM with a VIP label!
                            udm.add(item)
                        else:
                            best_gid, softmax_ratio, max_prob = gallery.classify_live_track(item['emb'], tau=TAU, exclude_gids=claimed_gids_this_frame)
                            if best_gid is not None and softmax_ratio >= TAU:
                                local_to_global_map[tid] = best_gid
                                claimed_gids_this_frame.add(best_gid)
                                gallery.add_observation(best_gid, item)
                                sighting_tracker.assign_global_id(tid, best_gid)
                            else:
                                udm.add(item)

        if frame_count % 30 == 0 and len(udm.pool) >= UDM_MIN_SIZE:
            actions = udm.run_dbscan(gallery)
            for act, gid, cnt, track_ids in actions:
                if act == "MINT":
                    create_person_document(gid, sighting_tracker._frame_to_ts(frame_count), persons_col)
                for tid in track_ids:
                    sighting_tracker.assign_global_id(tid, gid)
                    local_to_global_map[tid] = gid
                if act == "FACE_VIP_BYPASS":
                    tqdm.write(f"🎫 VIP Pass: Face ID {gid:03d} recognized... Pulling Body Tracks from UDM! ({track_ids})")
                else:
                    tqdm.write(f"UDM: {act} → GID {gid:03d} (Tracks: {track_ids})")

        pbar.update(1)

for tid in list(sighting_tracker.active_sightings.keys()):
    sighting_tracker.close_track(tid)

cap.release()
print(f"\n✅ Video Processing complete! Final Gallery: {gallery.num_classes()} distinct identities.")

### 📦 Cell 6: Multimodal CDN & Cloud Sync

In [ ]:
print("\n☁️ Pushing Face/Body Modalities to CDN, Pinecone, & MongoDB...")

def upload_and_index_sighting(sighting):
    gid = sighting['global_id']
    tid = sighting['track_id']
    target_folder = f"/supermarket_reid/global_id_{gid:03d}/"
    
    # 1. Upload Body (Using raw bytes to comply with ImageKit SDK update)
    img = sighting.get('best_crop_img')
    if img is not None:
        try:
            _, buffer = cv2.imencode('.jpg', img)
            filename = f"body_g{gid:03d}_t{tid}_{int(time.time())}.jpg"
            response = imagekit.files.upload(file=buffer.tobytes(), file_name=filename, folder=target_folder)
            sighting['best_crop_url'] = response.url if hasattr(response, 'url') else response.get('url', '')
        except:
            sighting['best_crop_url'] = ""
        del sighting['best_crop_img']

    # 2. Upload Face & Push to Pinecone "supermarket-faces" Index
    face_img = sighting.get('best_face_img')
    face_emb = sighting.get('best_face_emb')
    
    if face_img is not None and face_img.size > 0:
        try:
            _, f_buffer = cv2.imencode('.jpg', face_img)
            f_filename = f"face_g{gid:03d}_t{tid}_{int(time.time())}.jpg"
            f_response = imagekit.files.upload(file=f_buffer.tobytes(), file_name=f_filename, folder=target_folder)
            face_url = f_response.url if hasattr(f_response, 'url') else f_response.get('url', '')
            
            if face_emb is not None:
                vector_id = f"face_g{gid:03d}_{int(time.time())}"
                # Directly targeting the newly created 512D face index
                face_index.upsert(
                    vectors=[{"id": vector_id, "values": face_emb.tolist(), "metadata": {"global_id": gid, "url": face_url, "camera": sighting['camera_id']}}]
                )
            sighting['face_url'] = face_url
        except Exception as e:
            print(f"Face Error: {e}")
            sighting['face_url'] = ""
            
        del sighting['best_face_img']
        del sighting['best_face_emb']
        
    return sighting

completed_with_urls = []
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(upload_and_index_sighting, s) for s in sighting_tracker.completed_sightings if s.get('global_id') is not None]
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="CDN Sync"):
        completed_with_urls.append(future.result())

# Update MongoDB with Sightings and Faces (With strict float casting to prevent PyMongo crashes)
print("Updating MongoDB...")
for s in completed_with_urls:
    gid = s.pop('global_id') 
    face_url = s.pop('face_url', None)
    
    update_q = {
        "$push": {"sightings": s},
        "$set": {"last_seen_ts": float(s["exit_ts"])}
    }
    
    if face_url:
        update_q["$push"]["faces"] = {
            "url": face_url, 
            "timestamp": float(s["exit_ts"]), 
            "camera_id": str(s["camera_id"])
        }
        
    persons_col.update_one({"global_id": gid}, update_q)

print("Serializing True Output Buffer for Module 3 (α-FPS)...")
serializable = {}
for gid in gallery.gids:
    data = gallery.classes[gid]
    serializable[gid] = {
        "embeddings": [e.cpu().numpy() for e in data["embs"]],
        "ratios": data["ratios"],
        "det_scores": data["scores"],
        "filepaths": [str(p) for p in data["filepaths"]],
    }

with open(SAVE_BUFFER_PATH, 'wb') as f:
    pickle.dump(serializable, f)

shutil.make_archive('/kaggle/working/saved_identities', 'zip', str(DB_IMAGES_DIR))
print("✅ Done! Face and Body pipelines successfully linked and saved.")